# 00 — Dataset inventory
Run top to bottom after setting up the environment in `../README.md`. This notebook works with an empty dataset. It never downloads photos or modifies originals. Add images under `data/raw/<role>/<collection>/`, then rerun.

In [ ]:
from pathlib import Path
import sys

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'configs/baseline.json').is_file() and (p / 'src/adaptive_styles').is_dir()), None)
if root is None:
    raise RuntimeError('Start Jupyter from research/adaptive-styles.')
sys.path.insert(0, str(root / 'src'))
from adaptive_styles.dataset import inventory, write_inventory
print('Workspace:', root)

In [ ]:
from collections import Counter

rows = inventory(root)
print('Images:', len(rows))
print('Roles:', dict(Counter(row['role'] for row in rows)))
print('Collections:', dict(Counter((row['role'], row['collection_id']) for row in rows)))
print('Exact duplicate copies:', len(rows) - len({row['sha256'] for row in rows}))
if not rows:
    print('Waiting for your dataset. No synthetic samples or labels have been created.')

## Preview a few working images
Preview only: EXIF orientation is applied to the displayed copy. This is not color normalization for training; ICC/HDR conversion still needs its own preparation stage. RAW/HEIC formats may require a decoder beyond Pillow.

In [ ]:
from PIL import Image, ImageOps
from IPython.display import display

for row in rows[:6]:
    try:
        with Image.open(root / row['path']) as image:
            print(row['path'], image.size, image.mode, 'ICC:', bool(image.info.get('icc_profile')))
            preview = ImageOps.exif_transpose(image)
            preview.thumbnail((320, 240))
            display(preview.copy())
    except (OSError, ValueError) as error:
        print('Needs decoder or inspection:', row['path'], str(error))

## Save the inventory explicitly
Change `SAVE_INVENTORY` to `True` when ready. Existing output remains protected unless `OVERWRITE` is also enabled. Source and pair mappings use the CSV headers under `data/metadata/`; copy the `.example.csv` files to `sources.csv` and `pairs.csv` when adding real records.

In [ ]:
SAVE_INVENTORY = False
OVERWRITE = False
if SAVE_INVENTORY and rows:
    write_inventory(rows, root / 'data/processed/inventory.csv', overwrite=OVERWRITE)
    print('Saved inventory.')
else:
    print('No files written.')